### 朴素贝叶斯
朴素贝叶斯的前提是：
1. 每个特征变量在 **给定类别** 标签下是独立的
2. 每个变量在 **给定类别** 标签下服从正态分布

对于分类问题，朴素贝叶斯需要求得属于每个的概率，选择概率较大的类别作为预测结果
贝叶斯公式如下:
$$
P(y|x) = \frac{P(x|y)P(y)}{P(x)}
$$
全概率公式如下
$$
P(x) = \sum_{y}P(x|y)P(y)
$$
将贝叶斯公式变形为
$$
P(y|x) = \frac{P(x|y)P(y)}{P(x|y)P(y) + P(x|y^c)P(y^c)}
$$
对于贝叶斯公式来说
P(y)为先验概率 , 即在 x发生前对y的概率
p(y|x)为后验概率 , 即在 x发生后对y的概率
p(x|y)/p(x)为调整因子 , 对先验概率进行修正
由此可得 后验概率公式为:
$$
先验 * 调整因子 = 后验
$$
朴素贝叶斯的要求上面有体现，即为假设特征变量之间是独立的
即p(a|x)的时候,p(x)都为相同
所以只需要计算
$$
p(x|a)*p(a)
$$
并且因为特征之间是独立的
所以只需要计算
$$
p(x_1,x_2,x_3...|a)*p(a)
$$

In [27]:
from functools import reduce
import numpy as np
def create_data_set():
    posting_list=[['my', 'dog', 'has', 'flea', 'problems', 'help', 'please'],                #切分的词条
                 ['maybe', 'not', 'take', 'him', 'to', 'dog', 'park', 'stupid'],
                 ['my', 'dalmation', 'is', 'so', 'cute', 'I', 'love', 'him'],
                 ['stop', 'posting', 'stupid', 'worthless', 'garbage'],
                 ['mr', 'licks', 'ate', 'my', 'steak', 'how', 'to', 'stop', 'him'],
                 ['quit', 'buying', 'worthless', 'dog', 'food', 'stupid']]
    #1-代表侮辱性言论，0-代表正常
    class_vec = [0,1,0,1,0,1]
    return posting_list,class_vec
#词汇集合,即所有的词汇都应该出现在这个集合里
def create_vocab_list(data_set):
    vocab_set = set([])
    for document in data_set:
        #取并集
        vocab_set = vocab_set | set(document)
    return list(vocab_set)
def rule_words(vocab_list,input_set):
    vec = [0] * len(vocab_list)
    for word in input_set:
        if word in vocab_list:
            vec[vocab_list.index(word)] = 1
        else: print("the word: %s is not in my Vocabulary!" % word)
    return vec

def train_nb(train_matrix,train_category):
    num_doc = len(train_matrix)
    num_words = len(train_matrix[0])
    #计算侮辱文档的概率
    p_abusive = sum(train_category)/float(num_doc)
    #计算条件概率,即为当为侮辱文档的时候,返回的是一个向量组,每个词对应一个概率
    #这里是全概率公式
    word_count_abusive = np.zeros(num_words)
    word_count_no_abusive = np.zeros(num_words)
    count_abusive_denom = 0.0
    count_no_abusive_denom = 0.0
    for i in range(num_doc):
        if train_category[i] == 1:
            word_count_abusive+= train_matrix[i]
            count_abusive_denom += sum(train_matrix[i])
        else:
            word_count_no_abusive+= train_matrix[i]
            count_no_abusive_denom += sum(train_matrix[i])
    #该概率为了得到 p(x_1|a)，p(x_2|a)，.... ， p(x_n|a)
    p_count_abusive = word_count_abusive / count_abusive_denom
    p_count_no_abusive = word_count_no_abusive / count_no_abusive_denom
    #返回两个向量,他们分别为侮辱文档和正常文档的概率
    return p_abusive,p_count_abusive,p_count_no_abusive

def classify_nb(vec_test , vec_p_count_abusive , vec_p_count_no_abusive,p_abusive):
    vec_test = np.array(vec_test)
    #对应公式 p(a|x) = (p(x_1|a)*p(x_2|a)*....*p(x_n|a)) * p(a)
    p1 = reduce(lambda x,y:x*y,vec_p_count_abusive*vec_test) * p_abusive
    p0 = reduce(lambda x,y:x*y,vec_p_count_no_abusive*vec_test) * (1.0-p_abusive)
    print(p1)
    print(p0)
    if p1 > p0:
        return 1
    else:
        return 0

if __name__ == '__main__':
    data_set,class_vec = create_data_set()
    vocab_list = create_vocab_list(data_set)
    train = []
    for doc in data_set:
        train.append(rule_words(vocab_list,doc))
    pAb,p1V,p0V= train_nb(np.array(train),np.array(class_vec))
    test = np.array(rule_words(vocab_list,['love','my','dalmation']))
    if classify_nb(test,p1V,p0V,pAb) == 1:
        print('侮辱性')
    else:
        print('正常')

0.0
0.0
正常
